# FixtureDB: Fixture Patterns & Maintenance Indicators

This notebook examines fixture reuse and maintenance indicators:
- **Reuse count**: How many test functions use each fixture
- **Reuse vs complexity**: Relationship between fixture popularity and code structure
- **Teardown adoption**: Cleanup patterns across languages
- **Repository maturity impact**: How star tier and contributor count relate to fixture patterns

These analyses identify test infrastructure optimization opportunities.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

plt.rcParams['figure.facecolor'] = '#FAFAFA'
print("✓ Libraries imported")

In [ ]:
data_dir = Path('fixturedb')
repos = pd.read_csv(data_dir / 'repositories.csv')
fixtures = pd.read_csv(data_dir / 'fixtures.csv')

print(f"Repositories: {len(repos)}")
print(f"Fixtures: {len(fixtures)}")
print(f"\nAvailable columns: {', '.join(fixtures.columns.tolist())}")

## Fixture Reuse Distribution

In [ ]:
if 'reuse_count' in fixtures.columns:
    print("Fixture Reuse Count Statistics:")
    print(fixtures['reuse_count'].describe())
    
    fig, axs = plt.subplots(1, 2, figsize=(14, 5), facecolor='#FAFAFA')
    
    # Histogram
    axs[0].hist(fixtures['reuse_count'], bins=50, color='slateblue', edgecolor='black')
    axs[0].set_title('Fixture Reuse Count Distribution', fontsize=12, fontweight='bold')
    axs[0].set_xlabel('Reuse Count', fontsize=11)
    axs[0].set_ylabel('Number of Fixtures', fontsize=11)
    axs[0].grid(axis='y', alpha=0.3)
    
    # Cumulative
    reuse_sorted = np.sort(fixtures['reuse_count'])
    cumsum = np.cumsum(reuse_sorted) / reuse_sorted.sum()
    axs[1].plot(reuse_sorted, cumsum, linewidth=2, color='slateblue')
    axs[1].set_title('Cumulative Reuse Distribution', fontsize=12, fontweight='bold')
    axs[1].set_xlabel('Reuse Count (sorted)', fontsize=11)
    axs[1].set_ylabel('Cumulative Fraction', fontsize=11)
    axs[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nMedian reuse count: {fixtures['reuse_count'].median():.1f}")
    print(f"Fixtures reused 0 times: {(fixtures['reuse_count'] == 0).sum()} ({(fixtures['reuse_count'] == 0).sum() / len(fixtures) * 100:.1f}%)")
else:
    print("'reuse_count' column not available")

## Reuse Count vs Complexity

In [ ]:
if 'reuse_count' in fixtures.columns and 'cognitive_complexity' in fixtures.columns:
    fig, ax = plt.subplots(figsize=(10, 6), facecolor='#FAFAFA')
    
    ax.scatter(fixtures['reuse_count'], fixtures['cognitive_complexity'], 
               alpha=0.4, s=25, color='mediumseagreen', edgecolors='black', linewidth=0.3)
    ax.set_title('Fixture Reuse vs Complexity', fontsize=14, fontweight='bold')
    ax.set_xlabel('Reuse Count', fontsize=12)
    ax.set_ylabel('Cognitive Complexity', fontsize=12)
    ax.grid(alpha=0.3)
    
    corr = fixtures['reuse_count'].corr(fixtures['cognitive_complexity'])
    ax.text(0.95, 0.95, f'Pearson r = {corr:.3f}', 
            transform=ax.transAxes, ha='right', va='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    plt.tight_layout()
    plt.show()
else:
    print("Required columns not available")

## Teardown Adoption by Language

In [ ]:
if 'has_teardown_pair' in fixtures.columns:
    fixtures_with_lang = fixtures.merge(repos[['id', 'language']], left_on='repo_id', right_on='id', how='left')
    
    teardown_by_lang = fixtures_with_lang.groupby('language')['has_teardown_pair'].agg(['sum', 'count'])
    teardown_by_lang['adoption_rate'] = teardown_by_lang['sum'] / teardown_by_lang['count']
    
    print("Teardown Pair Adoption by Language:")
    print(teardown_by_lang)
    
    fig, ax = plt.subplots(figsize=(10, 6), facecolor='#FAFAFA')
    teardown_by_lang['adoption_rate'].plot(kind='bar', ax=ax, color=sns.color_palette('Set1', len(teardown_by_lang)))
    ax.set_title('Teardown Pair Adoption by Language', fontsize=14, fontweight='bold')
    ax.set_xlabel('Language', fontsize=12)
    ax.set_ylabel('Adoption Rate', fontsize=12)
    ax.set_ylim([0, 1])
    plt.xticks(rotation=45, ha='right')
    for i, v in enumerate(teardown_by_lang['adoption_rate']):
        ax.text(i, v + 0.02, f'{v*100:.1f}%', ha='center', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("'has_teardown_pair' column not available in public CSV (SQLite-only metric)")

## Repository Maturity Impact

In [ ]:
if 'num_contributors' in repos.columns and 'reuse_count' in fixtures.columns:
    fixtures_with_metadata = fixtures.merge(repos[['id', 'num_contributors', 'star_tier']], 
                                             left_on='repo_id', right_on='id', how='left')
    
    fig, axs = plt.subplots(1, 2, figsize=(14, 5), facecolor='#FAFAFA')
    
    # By contributors
    fixtures_with_metadata.boxplot(column='reuse_count', by='num_contributors', ax=axs[0], patch_artist=True)
    axs[0].set_title('Reuse Count vs Contributor Count', fontsize=12, fontweight='bold')
    axs[0].set_xlabel('Contributor Count Bucket', fontsize=11)
    axs[0].set_ylabel('Fixture Reuse Count', fontsize=11)
    plt.suptitle('')
    
    # By star tier
    fixtures_with_metadata.boxplot(column='reuse_count', by='star_tier', ax=axs[1], patch_artist=True)
    axs[1].set_title('Reuse Count vs Star Tier', fontsize=12, fontweight='bold')
    axs[1].set_xlabel('Star Tier', fontsize=11)
    axs[1].set_ylabel('Fixture Reuse Count', fontsize=11)
    
    plt.tight_layout()
    plt.show()
else:
    print("Required columns not available")

## Key Findings

**Fixture Reuse Patterns:**
- Most fixtures see low reuse (median < 1), indicating project-specific setups
- Reused fixtures tend to be simpler (lower complexity for higher reuse)
- Teardown adoption varies by language convention
- Repository maturity (contributors, star tier) correlates with fixture patterns

**Optimization Opportunities:**
- Low-reuse but complex fixtures may benefit from refactoring
- Variance in teardown adoption suggests standardization opportunities
- Mature projects show stronger fixture consolidation patterns